# Membership Inference Attacks on ML Models
## Privacy Auditing and GDPR Implications

**Author:** Jonathan Ted Benson Cerullo Uyi  
**Course:** Ethics in Artificial Intelligence — University of Bologna, MSc AI  

---

### How to use this notebook
Run cells **top to bottom**, in order. Each section is independent but builds on the previous ones.



## Abstract

This notebook implements a privacy auditing pipeline for machine learning models,
demonstrating how **Membership Inference Attacks (MIA)** can reveal whether specific
data points were used during training. We evaluate two attack strategies
(threshold-based and shadow model) across four defense configurations:

1. **Overfitted model** — worst-case privacy scenario (no defenses)
2. **Regularised model** — informal defenses (L2 + Dropout + Early Stopping)
3. **DP-SGD** — formal differential privacy guarantees (ε = 10, 5, 1)

We use **CIFAR-10** as a controlled benchmark, following the methodology established
by Shokri et al. (2017) and Carlini et al. (2022). Results are mapped to
**GDPR Articles 5, 6, 17, and 32** to assess the regulatory implications of
membership inference vulnerabilities.

### Why CIFAR-10?

During initial development, empirical analysis on the Adult Census Income dataset
showed that MIA yields near-random AUC (~0.57) even under extreme overfitting,
due to the low-dimensional, highly regular nature of tabular data. We adopted
CIFAR-10 — a standard MIA benchmark — where membership inference is empirically
feasible and allows meaningful evaluation of DP-SGD defenses. The privacy
implications extend to any domain where ML models are trained on personal data:
medical imaging, biometric recognition, and surveillance systems.

---
## 0. Install dependencies

In [ ]:
# Run once to install all dependencies
!pip install -q torch torchvision opacus scikit-learn pandas matplotlib datasets
print("All dependencies installed.")


---
## 1. Imports and global configuration

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import json
import warnings
import os

from pathlib import Path
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Reproducibility (CPU + GPU + cuDNN)
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Output dirs
Path("results").mkdir(exist_ok=True)
Path("figures").mkdir(exist_ok=True)

# Plot style
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
BLUE, ORANGE, RED, GREY = "#1f77b4", "#ff7f0e", "#d62728", "#7f7f7f"

print("Setup complete.")


---
## 2. Data loading and preprocessing

We load CIFAR-10 (60,000 images, 10 classes) and create **mutually disjoint
splits** drawn from the same underlying training set, so no sample is ever
shared between the target model, the shadow models, the regularised
validation set, and the DP-SGD training pool:

- **target_train** (2,500): training data for the target model (members)
- **target_test** (2,500): held-out non-members (drawn from CIFAR-10 test)
- **shadow_train** (5,000): training data for shadow models
- **shadow_test** (2,500): non-members for shadow models
- **reg_val** (500): validation set for early stopping (regularised model)
- **dp_train** (15,000): training pool for DP-SGD experiments. **Includes
  `target_train`** (so the 2,500 target members are real members of the DP
  model, enabling a fair MIA attack), but is **disjoint from `shadow_train`,
  `shadow_test`, and `reg_val`** (so the shadow attack remains
  methodologically valid).

The smaller target training set (2,500) creates a realistic scenario where
overfitting — and thus privacy leakage — is more pronounced. This mirrors
real-world settings with limited data (medical imaging, HR records).


In [ ]:
# CIFAR-10 via Hugging Face Datasets
# We bypass torchvision's hardcoded toronto.edu mirror (intermittently down)
# and load CIFAR-10 from the Hugging Face Hub instead. The resulting numpy
# arrays match the format that torchvision.datasets.CIFAR10 originally
# produced, so no other cell needs to change.
import torchvision.transforms as T
from datasets import load_dataset

try:
    ds = load_dataset("uoft-cs/cifar10")
except Exception as e:
    print(f"  primary load failed: {e}; trying legacy alias 'cifar10'")
    ds = load_dataset("cifar10")

# Detect column names (older revisions used "img"/"label", a few mirrors
# use "image"/"fine_label"). Fall back provided.
def _detect_columns(split):
    cols = set(split.column_names)
    img_col   = "img"   if "img"   in cols else ("image" if "image" in cols else None)
    label_col = "label" if "label" in cols else ("fine_label" if "fine_label" in cols else None)
    if img_col is None or label_col is None:
        raise RuntimeError(
            f"Unrecognised CIFAR-10 schema: columns={split.column_names}"
        )
    return img_col, label_col

img_col, label_col = _detect_columns(ds["train"])

def _hf_to_numpy(split, img_col, label_col):
    """Convert a HF split to (uint8 numpy array, int64 numpy array).

    Output shape: (N, 32, 32, 3) for images, (N,) for labels - identical to
    what `torchvision.datasets.CIFAR10(...).data` and `.targets` would return.
    """
    n = len(split)
    X = np.zeros((n, 32, 32, 3), dtype=np.uint8)
    y = np.zeros(n, dtype=np.int64)
    for i in range(n):
        ex = split[i]
        X[i] = np.asarray(ex[img_col], dtype=np.uint8)
        y[i] = ex[label_col]
    return X, y

X_all_train, y_all_train = _hf_to_numpy(ds["train"], img_col, label_col)
X_all_test,  y_all_test  = _hf_to_numpy(ds["test"],  img_col, label_col)

print(f"CIFAR-10 loaded from HF: {len(X_all_train):,} train, {len(X_all_test):,} test")
print(
    f"Classes: {len(np.unique(y_all_train))} "
    f"- Image shape: {X_all_train.shape[1:]}"
)

# Disjoint splits
TRAIN_SIZE = 2_500          # target model training (members)
TEST_SIZE = 2_500           # held-out non-members (from CIFAR-10 test)
SHADOW_TRAIN_SIZE = 5_000   # shadow models' training pool
SHADOW_TEST_SIZE = 2_500    # shadow models' non-members pool
REG_VAL_SIZE = 500          # validation set for early stopping (reg. model)
DP_TRAIN_SIZE = 15_000      # DP-SGD training pool (includes target_train,
                            # disjoint from shadow_*, reg_val)

rng = np.random.default_rng(SEED)
idx_train = rng.permutation(len(X_all_train))

a = TRAIN_SIZE                       # 2_500
b = a + SHADOW_TRAIN_SIZE            # 7_500
c = b + SHADOW_TEST_SIZE             # 10_000
d = c + REG_VAL_SIZE                 # 10_500
extra = DP_TRAIN_SIZE - TRAIN_SIZE   # 12_500
e = d + extra                        # 23_000
assert e <= len(idx_train), (
    f"Need {e} indices, have {len(idx_train)}."
)

ti       = idx_train[0:a]    # target_train
si       = idx_train[a:b]    # shadow_train
st       = idx_train[b:c]    # shadow_test
rv       = idx_train[c:d]    # reg_val (regularized validation)
dp_extra = idx_train[d:e]    # extra DP pool (disjoint from ti/si/st/rv)
dp_idx   = np.concatenate([ti, dp_extra])  # DP target set: ti + extra

# Sanity checks: dp_idx must include ti but be disjoint from si/st/rv.
assert len(np.intersect1d(si, dp_idx)) == 0, "si and dp_idx overlap"
assert len(np.intersect1d(st, dp_idx)) == 0, "st and dp_idx overlap"
assert len(np.intersect1d(rv, dp_idx)) == 0, "rv and dp_idx overlap"
assert len(np.intersect1d(ti, dp_idx)) == TRAIN_SIZE, (
    "ti must be a subset of dp_idx"
)
# ti, si, st, rv are pairwise disjoint by construction (slices).

idx_test = rng.permutation(len(X_all_test))
tt = idx_test[:TEST_SIZE]

X_tr,    y_tr    = X_all_train[ti], y_all_train[ti]
X_te,    y_te    = X_all_test[tt],  y_all_test[tt]
X_sh_tr, y_sh_tr = X_all_train[si], y_all_train[si]
X_sh_te, y_sh_te = X_all_train[st], y_all_train[st]
X_rv,    y_rv    = X_all_train[rv], y_all_train[rv]
X_tr_dp, y_tr_dp = X_all_train[dp_idx], y_all_train[dp_idx]

print()
print("Disjoint splits:")
print(f"  target_train = {len(X_tr):,} (members)")
print(f"  target_test  = {len(X_te):,} (non-members)")
print(f"  shadow_train = {len(X_sh_tr):,}")
print(f"  shadow_test  = {len(X_sh_te):,}")
print(f"  reg_val      = {len(X_rv):,}  (early-stopping validation)")
print(f"  dp_train     = {len(X_tr_dp):,} (includes target_train)")
print("Data ready (all splits are disjoint as required).")


---
## 3. Model Architecture & Training Utilities

We use a compact CNN (4 conv layers + 2 FC layers, ~1.05M parameters) designed
to overfit on small datasets. The architecture is intentionally simple — this is
not a benchmark for accuracy but a **privacy auditing experiment**.

Key training modes:
- `train_standard`: vanilla SGD, no privacy protections
- `train_with_early_stopping`: validation-based stopping as informal defense
- **DP-SGD** (via Opacus): per-sample gradient clipping + calibrated noise injection

In [ ]:
# PyTorch Dataset (images)
class ImageDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = torch.tensor(y, dtype=torch.long)
        self.transform = transform

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        img = self.X[i]
        if self.transform:
            from PIL import Image
            img = Image.fromarray(img)
            img = self.transform(img)
        else:
            img = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1) / 255.0
        return img, self.y[i]

TRANSFORM_TRAIN = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomCrop(32, padding=4),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2399, 0.2010)),
])

TRANSFORM_TEST = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2399, 0.2010)),
])

def get_loaders(X_train, y_train, X_test, y_test, batch_size=128):
    return (
        DataLoader(
            ImageDataset(X_train, y_train, TRANSFORM_TRAIN),
            batch_size=batch_size, shuffle=True,
        ),
        DataLoader(
            ImageDataset(X_test, y_test, TRANSFORM_TEST),
            batch_size=256, shuffle=False,
        ),
    )

# CNN Architecture
class CNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.0):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 256), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# Training
def train_standard(model, loader, epochs=30, lr=1e-3, weight_decay=1e-4):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    crit = nn.CrossEntropyLoss()
    for ep in range(1, epochs + 1):
        model.train()
        loss_sum = 0
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            l = crit(model(X), y)
            l.backward()
            opt.step()
            loss_sum += l.item() * len(y)
        if ep % 30 == 0 or ep == epochs:
            print(
                f"  epoch {ep:3d}/{epochs} "
                f"loss={loss_sum / len(loader.dataset):.4f}"
            )

def train_with_early_stopping(model, train_loader, val_loader,
                              epochs=100, lr=1e-3, weight_decay=1e-4,
                              patience=10):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    crit = nn.CrossEntropyLoss()
    best_val_acc = 0.0
    best_state = None
    wait = 0

    for ep in range(1, epochs + 1):
        model.train()
        loss_sum = 0
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            l = crit(model(X), y)
            l.backward()
            opt.step()
            loss_sum += l.item() * len(y)

        val_acc = evaluate(model, val_loader)

        if ep % 10 == 0:
            print(
                f"  epoch {ep:3d}/{epochs} "
                f"loss={loss_sum / len(train_loader.dataset):.4f} "
                f"val_acc={val_acc:.3f}"
            )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(
                    f"  Early stopping at epoch {ep} "
                    f"(best val_acc={best_val_acc:.3f})"
                )
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        correct += (model(X).argmax(1) == y).sum().item()
        total += len(y)
    return correct / total

@torch.no_grad()
def confidence_scores(model, loader):
    model.eval()
    probs = []
    for X, _ in loader:
        probs.append(torch.softmax(model(X.to(DEVICE)), dim=1).cpu())
    return torch.cat(probs)

# Attack metrics
def tpr_at_fpr(y_true, scores, target_fpr=0.01):
    fpr, tpr, _ = roc_curve(y_true, scores)
    return float(np.interp(target_fpr, fpr, tpr))

def print_attack(name, y_true, scores):
    """Compute and print MIA evaluation metrics.

    Returns a dict with privacy-relevant scalars, all cast to native floats so
    json.dump works without TypeError. The legacy 'accuracy' field has been
    removed: at the median threshold it is a deterministic function of AUC and
    so does not provide additional information for MIA evaluation.
    """
    auc = float(roc_auc_score(y_true, scores))
    t1 = float(tpr_at_fpr(y_true, scores, 0.01))
    t01 = float(tpr_at_fpr(y_true, scores, 0.001))
    print(
        f"  [{name:13s}] AUC={auc:.3f} "
        f"TPR@1%FPR={t1:.3f} TPR@0.1%FPR={t01:.3f}"
    )
    return {"auc": auc, "tpr_at_1_fpr": t1, "tpr_at_01_fpr": t01}

print("Helper functions defined.")


---
## 4a. Attack implementations

### 1) Threshold Attack (Yeom et al., 2018)
Uses **per-sample loss** as membership signal. Intuition: a model that memorized
its training data will have significantly lower loss on members than non-members.

### 2) Shadow Model Attack (Shokri et al., 2017)
Trains multiple shadow models to learn a **meta-classifier** that predicts
membership from model behavior. Features used:
- Max confidence, correct-class confidence
- Prediction entropy, per-sample loss (raw and log-scale)
- Confidence gap (difference between top-2 predictions)

The meta-classifier (Gradient Boosting) captures non-linear patterns in these
features that correlate with membership status.

In [ ]:
def threshold_attack(model, member_loader, nonmember_loader):
    """Loss-based threshold attack (Yeom et al., 2018)."""
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction='none')

    def get_losses(loader):
        losses = []
        with torch.no_grad():
            for X, y in loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                loss = criterion(model(X), y).cpu().numpy()
                losses.append(loss)
        return np.concatenate(losses)

    m_losses = get_losses(member_loader)
    n_losses = get_losses(nonmember_loader)

    scores = np.concatenate([-m_losses, -n_losses])   # low loss = member  ->  high score = member
    labels = np.array([1] * len(m_losses) + [0] * len(n_losses))
    return print_attack("Threshold", labels, scores)

def shadow_model_attack(target_model, member_loader, nonmember_loader,
                        X_sh_tr, y_sh_tr, X_sh_te, y_sh_te,
                        n_shadows=4, shadow_epochs=100):
    """Shadow model attack with per-sample loss + confidence features."""
    rng2 = np.random.default_rng(SEED)
    meta_X, meta_y = [], []
    criterion = nn.CrossEntropyLoss(reduction='none')

    def extract_features_from_loader(model, loader):
        model.eval()
        all_feats = []
        with torch.no_grad():
            for Xb, yb in loader:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                logits = model(Xb)
                probs = torch.softmax(logits, dim=1).cpu().numpy()
                losses = criterion(logits, yb).cpu().numpy()

                max_conf = probs.max(axis=1)
                correct_conf = probs[np.arange(len(yb)), yb.cpu().numpy()]
                entropy = -(probs * np.log(probs + 1e-10)).sum(axis=1)
                log_loss = np.log(losses + 1e-10)
                sorted_probs = np.sort(probs, axis=1)
                conf_gap = sorted_probs[:, -1] - sorted_probs[:, -2]

                feats = np.stack([
                    max_conf, correct_conf, entropy,
                    losses, log_loss, conf_gap
                ], axis=1)
                all_feats.append(feats)
        return np.vstack(all_feats)

    for i in range(n_shadows):
        print(f"  Training shadow model {i+1}/{n_shadows}...")
        n_in = len(X_sh_tr)
        n_out = len(X_sh_te)
        perm_in = rng2.permutation(n_in)
        perm_out = rng2.permutation(n_out)
        half = min(n_in, n_out) // 2

        idx_in = perm_in[:half]
        idx_out = perm_out[:half]

        X_s_in, y_s_in = X_sh_tr[idx_in], y_sh_tr[idx_in]
        X_s_out, y_s_out = X_sh_te[idx_out], y_sh_te[idx_out]

        sh_tr_loader, _ = get_loaders(X_s_in, y_s_in, X_s_out, y_s_out)
        shadow = CNN().to(DEVICE)
        train_standard(shadow, sh_tr_loader,
                       epochs=shadow_epochs, weight_decay=0.0)

        in_loader = DataLoader(
            ImageDataset(X_s_in, y_s_in, TRANSFORM_TEST),
            batch_size=256, shuffle=False,
        )
        out_loader = DataLoader(
            ImageDataset(X_s_out, y_s_out, TRANSFORM_TEST),
            batch_size=256, shuffle=False,
        )

        in_feats = extract_features_from_loader(shadow, in_loader)
        out_feats = extract_features_from_loader(shadow, out_loader)

        meta_X.append(in_feats)
        meta_X.append(out_feats)
        meta_y.append(np.ones(len(in_feats), dtype=int))
        meta_y.append(np.zeros(len(out_feats), dtype=int))

    meta_X = np.vstack(meta_X)
    meta_y = np.concatenate(meta_y)

    clf = GradientBoostingClassifier(
        n_estimators=100, max_depth=3, random_state=SEED
    )
    clf.fit(meta_X, meta_y)

    m_feats = extract_features_from_loader(target_model, member_loader)
    n_feats = extract_features_from_loader(target_model, nonmember_loader)
    all_feats = np.vstack([m_feats, n_feats])
    labels = np.array([1] * len(m_feats) + [0] * len(n_feats))
    scores = clf.predict_proba(all_feats)[:, 1]

    return print_attack("Shadow Model", labels, scores)

print("✅ Attack functions defined.")

## 4b. Shared Loaders

In [ ]:
train_loader, test_loader = get_loaders(X_tr, y_tr, X_te, y_te)

member_loader = DataLoader(
    ImageDataset(X_tr, y_tr, TRANSFORM_TEST),
    batch_size=256, shuffle=False,
)
nonmember_loader = DataLoader(
    ImageDataset(X_te, y_te, TRANSFORM_TEST),
    batch_size=256, shuffle=False,
)

RESULTS = {"dp": []}
print("✅ Shared loaders ready.")

---
## 5. Experiments

We evaluate **5 configurations**:

| # | Config | Purpose |
|---|--------|---------|
| 1 | Overfitted | Worst-case privacy scenario |
| 2 | Regularised | Informal defence (L2 + Dropout + Early Stopping) |
| 3 | DP-SGD ε=10 | Formal defence, mild noise |
| 4 | DP-SGD ε=5 | Formal defence, moderate noise |
| 5 | DP-SGD ε=1 | Formal defence, strong noise |

Results are saved to `results/results.json`.

### Experiment 1: Overfitted Model (Worst Case)

**Goal**: Establish the upper bound of privacy leakage.

Settings: 150 epochs, no regularisation, no dropout — the model is free to
memorize the training data completely. We expect:
- High generalization gap (>30%)
- Attack AUC significantly above 0.5
- This represents the **worst-case scenario** for data subjects whose records
 are in the training set.

In [ ]:
print("=" * 55)
print("Experiment 1: Overfitted model (150 epochs, no regularisation)")
print("=" * 55)

model_overfit = CNN(dropout=0.0)
train_standard(model_overfit, train_loader, epochs=150, weight_decay=0.0)

tr_acc = evaluate(model_overfit, member_loader)
te_acc = evaluate(model_overfit, test_loader)
print(f"\nTrain acc={tr_acc:.3f} Test acc={te_acc:.3f} "
      f"Gen. gap={tr_acc - te_acc:.3f}")

print("\nAttack results:")
thr = threshold_attack(model_overfit, member_loader, nonmember_loader)
shd = shadow_model_attack(
    model_overfit, member_loader, nonmember_loader,
    X_sh_tr, y_sh_tr, X_sh_te, y_sh_te,
    shadow_epochs=100,
)

RESULTS["overfitted"] = {
    "train_acc": tr_acc, "test_acc": te_acc,
    "gen_gap": tr_acc - te_acc,
    "threshold": thr, "shadow": shd,
}
print("\n✅ Experiment 1 done.")

### Experiment 2: Regularised Model (Informal Defense)

**Goal**: Measure whether standard ML best practices reduce privacy leakage.

Settings: L2 regularisation (weight_decay=1e-3), Dropout (p=0.5),
Early Stopping (patience=10). These are techniques commonly used for better
generalisation — we assess their **side effect** on privacy.

Note: These are **not** formal privacy guarantees. A regularised model may still
leak information, just less than a completely overfitted one.

In [ ]:
print("=" * 55)
print("Experiment 2: Regularised (L2 + Dropout=0.5 + Early Stopping)")
print("=" * 55)

# We train the regularised model on the FULL target_train (X_tr, 2,500
# samples) so the member set used for MIA is identical to Experiment 1.
# Early stopping uses a separate, disjoint validation set (X_rv, 500
# samples) - this fixes a methodological bias of an earlier draft, where
# 20% of X_tr was held out for validation but still treated as "members"
# during the attack.
reg_train_loader, _ = get_loaders(X_tr, y_tr, X_rv, y_rv)
val_loader = DataLoader(
    ImageDataset(X_rv, y_rv, TRANSFORM_TEST),
    batch_size=256, shuffle=False,
)

model_reg = CNN(dropout=0.5)
train_with_early_stopping(
    model_reg, reg_train_loader, val_loader,
    epochs=100, weight_decay=1e-3, patience=10,
)

tr_acc = evaluate(model_reg, member_loader)
te_acc = evaluate(model_reg, test_loader)
print()
print(
    f"Train acc={tr_acc:.3f} Test acc={te_acc:.3f} "
    f"Gen. gap={tr_acc - te_acc:.3f}"
)

print()
print("Attack results:")
thr = threshold_attack(model_reg, member_loader, nonmember_loader)
shd = shadow_model_attack(
    model_reg, member_loader, nonmember_loader,
    X_sh_tr, y_sh_tr, X_sh_te, y_sh_te,
    shadow_epochs=100,
)

RESULTS["regularised"] = {
    "train_acc": tr_acc, "test_acc": te_acc,
    "gen_gap": tr_acc - te_acc,
    "threshold": thr, "shadow": shd,
}
print()
print("Experiment 2 done.")


### Experiment 3: DP-SGD (Formal Privacy Guarantee)

**Goal**: Evaluate the only defense that provides **mathematical privacy guarantees**.

DP-SGD (Abadi et al., 2016) modifies standard SGD by:
1. **Clipping** per-sample gradients to bound sensitivity
2. **Adding calibrated Gaussian noise** to the aggregated gradient
3. **Tracking privacy budget** (ε) via Rényi Differential Privacy accounting

We test three privacy levels:
- **ε=10**: relaxed privacy (more utility, some protection)
- **ε=5**: moderate privacy
- **ε=1**: strong privacy (significant utility cost)

DP experiments use a larger training set (15,000 samples) because DP-SGD
requires more data to overcome the noise injected during training — a key
practical consideration for organisations adopting this technique.

> ⚠️ These cells require **Opacus** and run slower than standard training.

In [ ]:
try:
    from opacus import PrivacyEngine
    OPACUS_OK = True
    print("Opacus available.")
except ImportError:
    OPACUS_OK = False
    print("Opacus not found.")

# sweep parameters
DELTA = 1e-5
EPSILON_VALUES = [10, 5, 1]

if not OPACUS_OK:
    print("Skipping DP experiments - install Opacus first.")
else:
    for target_eps in EPSILON_VALUES:  # for each epsilon we have the full experiment (training, evaluation, attack)
        print("=" * 55)
        print(f"DP-SGD eps={target_eps} delta={DELTA}")
        print("=" * 55)

        dp_train_loader, dp_test_loader = get_loaders(
            X_tr_dp, y_tr_dp, X_te, y_te
        )

        model_dp = CNN(dropout=0.0).to(DEVICE)
        optimizer = torch.optim.Adam(model_dp.parameters(), lr=2e-3)
        criterion = nn.CrossEntropyLoss()

        privacy_engine = PrivacyEngine() # object that handles the DP-SGD process (instance of PrivateEngine from Opacus)

        DP_EPOCHS = 60

        result = privacy_engine.make_private_with_epsilon(
            module=model_dp,
            optimizer=optimizer,
            data_loader=dp_train_loader,
            epochs=DP_EPOCHS,
            target_epsilon=target_eps,
            target_delta=DELTA,
            max_grad_norm=1.2,
        )
        model_dp, optimizer, dp_train_loader = result

        for ep in range(1, DP_EPOCHS + 1):
            model_dp.train()
            for X, y in dp_train_loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                optimizer.zero_grad()
                criterion(model_dp(X), y).backward()
                optimizer.step()
            if ep % 15 == 0 or ep == DP_EPOCHS:
                eps_spent = privacy_engine.get_epsilon(DELTA)
                print(f"  epoch {ep:3d}/{DP_EPOCHS} eps_spent={eps_spent:.2f}")

        actual_eps = float(privacy_engine.get_epsilon(DELTA))

        raw_model = (
            model_dp._module if hasattr(model_dp, "_module") else model_dp
        )
        raw_model.to(DEVICE)

        # member_acc = accuracy on the 2,500 target members (a subset of the
        # DP training set - see Cell 7). non_member_acc = accuracy on X_te
        # (held-out CIFAR-10 test). The difference is the generalisation gap
        # of the DP model on the MIA evaluation prism.
        member_acc = evaluate(raw_model, DataLoader(
            ImageDataset(X_tr, y_tr, TRANSFORM_TEST),
            batch_size=256, shuffle=False,
        ))
        non_member_acc = evaluate(raw_model, nonmember_loader)
        print()
        print(
            f"eps_actual={actual_eps:.2f} "
            f"member_acc={member_acc:.3f} non_member_acc={non_member_acc:.3f} "
            f"Gen. gap={member_acc - non_member_acc:.3f}"
        )

        print()
        print("Attack results:")
        # The attack uses the original 2,500 members for fair comparison with
        # Experiments 1 and 2.
        dp_attack_member = DataLoader(
            ImageDataset(X_tr[:TRAIN_SIZE], y_tr[:TRAIN_SIZE], TRANSFORM_TEST),
            batch_size=256, shuffle=False,
        )
        thr = threshold_attack(
            raw_model, dp_attack_member, nonmember_loader
        )
        shd = shadow_model_attack(
            raw_model, dp_attack_member, nonmember_loader,
            X_sh_tr, y_sh_tr, X_sh_te, y_sh_te,
            shadow_epochs=100,
        )

        RESULTS["dp"].append({
            "epsilon_target": target_eps,
            "epsilon_actual": actual_eps,
            "delta": DELTA,
            "train_acc": member_acc,
            "test_acc": non_member_acc,
            "gen_gap": member_acc - non_member_acc,
            "threshold": thr,
            "shadow": shd,
        })

        # Store model weights for ROC plotting later
        if not hasattr(torch, "_dp_models"):
            torch._dp_models = {}
        torch._dp_models[target_eps] = {
            k: v.cpu().clone() for k, v in raw_model.state_dict().items()
        }

        with open("results/results.json", "w") as f:
            json.dump(RESULTS, f, indent=2)

        print()
        print(f"DP eps={target_eps} done.")
        print()

    with open("results/results.json", "w") as f:
        json.dump(RESULTS, f, indent=2)
    print("All results saved to results/results.json")


---
## 6. Results summary table

In [ ]:
rows = []

for key in ["overfitted", "regularised"]:
    if key not in RESULTS:
        continue
    cfg = RESULTS[key]
    label = "Overfitted" if key == "overfitted" else "Regularised"
    rows.append({
        "Config": label,
        "eps": "inf",
        "Train acc": f"{cfg['train_acc']:.3f}",
        "Test acc": f"{cfg['test_acc']:.3f}",
        "Gen gap": f"{cfg['gen_gap']:.3f}",
        "Thr AUC": f"{cfg['threshold']['auc']:.3f}",
        "Shd AUC": f"{cfg['shadow']['auc']:.3f}",
        "TPR@1%FPR (shadow)": f"{cfg['shadow']['tpr_at_1_fpr']:.3f}",
    })

for dp in RESULTS.get("dp", []):
    rows.append({
        "Config": "DP-SGD",
        "eps": f"{dp['epsilon_target']}",
        "Train acc": f"{dp['train_acc']:.3f}",
        "Test acc": f"{dp['test_acc']:.3f}",
        "Gen gap": f"{dp['gen_gap']:.3f}",
        "Thr AUC": f"{dp['threshold']['auc']:.3f}",
        "Shd AUC": f"{dp['shadow']['auc']:.3f}",
        "TPR@1%FPR (shadow)": f"{dp['shadow']['tpr_at_1_fpr']:.3f}",
    })

df_res = pd.DataFrame(rows)
print(df_res.to_string(index=False))


---
## 7. Figures

In [ ]:
plot_configs = []
if "overfitted" in RESULTS:
    plot_configs.append(("Overfitted", RESULTS["overfitted"]))
if "regularised" in RESULTS:
    plot_configs.append(("Regularised", RESULTS["regularised"]))
for dp in RESULTS.get("dp", []):
    plot_configs.append((f"DP ε={dp['epsilon_target']}", dp))

labels = [c[0] for c in plot_configs]
configs = [c[1] for c in plot_configs]
x = np.arange(len(labels))
print(f"Plotting {len(labels)} configurations: {labels}")

# Figure 1 — Generalisation gap
train_accs = [c["train_acc"] for c in configs]
test_accs = [c["test_acc"] for c in configs]
gaps = [c["gen_gap"] for c in configs]

w = 0.25
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w, train_accs, w, label="Train acc", color=BLUE, alpha=0.85)
ax.bar(x, test_accs, w, label="Test acc", color=ORANGE, alpha=0.85)
ax.bar(x + w, gaps, w, label="Gen. gap", color=RED, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_title("Generalisation Gap Across Configurations")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig("figures/01_gen_gap.pdf", bbox_inches="tight")
plt.show()
print("Saved figures/01_gen_gap.pdf")

# Figure 2 — Attack AUC
thr_aucs = [c["threshold"]["auc"] for c in configs]
shd_aucs = [c["shadow"]["auc"] for c in configs]

w = 0.35
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w / 2, thr_aucs, w, label="Threshold attack", color=BLUE, alpha=0.85)
ax.bar(x + w / 2, shd_aucs, w, label="Shadow model", color=ORANGE, alpha=0.85)
ax.axhline(0.5, color=GREY, linestyle="--", linewidth=1, label="Random (AUC=0.5)")
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylim(0.45, 1.0)
ax.set_ylabel("AUC")
ax.set_title("Attack AUC per Configuration")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig("figures/02_attack_auc.pdf", bbox_inches="tight")
plt.show()
print("Saved figures/02_attack_auc.pdf")

# Figure 3 — TPR@1%FPR
tpr1_thr = [c["threshold"]["tpr_at_1_fpr"] for c in configs]
tpr1_shd = [c["shadow"]["tpr_at_1_fpr"] for c in configs]

w = 0.35
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w / 2, tpr1_thr, w, label="Threshold attack", color=BLUE, alpha=0.85)
ax.bar(x + w / 2, tpr1_shd, w, label="Shadow model", color=ORANGE, alpha=0.85)
ax.axhline(0.01, color=GREY, linestyle="--", linewidth=1,
           label="Random baseline (0.01)")
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("TPR @ 1% FPR")
ax.set_title("Privacy Leakage: TPR at 1% FPR")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig("figures/03_tpr_fpr.pdf", bbox_inches="tight")
plt.show()
print("Saved figures/03_tpr_fpr.pdf")

# Figure 4 — Privacy–utility trade-off
dp_cfgs = RESULTS.get("dp", [])
if not dp_cfgs:
    print("No DP results yet — run the DP cells first.")
else:
    eps_v = [c["epsilon_actual"] for c in dp_cfgs]
    t_accs = [c["test_acc"] for c in dp_cfgs]
    tprs = [c["shadow"]["tpr_at_1_fpr"] for c in dp_cfgs]

    if "regularised" in RESULTS:
        eps_v.append(1000)
        t_accs.append(RESULTS["regularised"]["test_acc"])
        tprs.append(RESULTS["regularised"]["shadow"]["tpr_at_1_fpr"])

    order = np.argsort(eps_v)
    eps_v = np.array(eps_v)[order]
    t_accs = np.array(t_accs)[order]
    tprs = np.array(tprs)[order]

    fig, ax1 = plt.subplots(figsize=(7, 4))
    ax1.plot(range(len(eps_v)), t_accs, "o-", color=BLUE,
             label="Test accuracy")
    ax1.set_ylabel("Test accuracy", color=BLUE)
    ax1.tick_params(axis="y", labelcolor=BLUE)

    ax2 = ax1.twinx()
    ax2.plot(range(len(eps_v)), tprs, "s--", color=RED,
             label="TPR@1%FPR (shadow)")
    ax2.set_ylabel("TPR @ 1% FPR", color=RED)
    ax2.tick_params(axis="y", labelcolor=RED)
    ax2.spines["right"].set_visible(True)

    xtick_labels = [
        str(int(e)) if e < 100 else "∞ (no DP)" for e in eps_v
    ]
    ax1.set_xticks(range(len(eps_v)))
    ax1.set_xticklabels(xtick_labels)
    ax1.set_xlabel("Privacy budget ε")

    lines1, lbls1 = ax1.get_legend_handles_labels()
    lines2, lbls2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, lbls1 + lbls2,
               frameon=False, loc="lower right")
    ax1.set_title("Privacy–Utility Trade-off under DP-SGD")
    fig.tight_layout()
    fig.savefig("figures/04_privacy_utility.pdf", bbox_inches="tight")
    plt.show()
    print("Saved figures/04_privacy_utility.pdf")

# ── Figure 5 — ROC Curves Overlay ────────────────────────────────────────────
from sklearn.metrics import roc_curve, roc_auc_score

criterion_roc = nn.CrossEntropyLoss(reduction='none')

def get_mia_roc_data(model, m_loader, nm_loader):
    """Get loss-based MIA scores and labels for ROC plotting."""
    model.eval()

    def get_losses(loader):
        losses = []
        with torch.no_grad():
            for X, y in loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                loss = criterion_roc(model(X), y).cpu().numpy()
                losses.append(loss)
        return np.concatenate(losses)

    m_losses = get_losses(m_loader)
    n_losses = get_losses(nm_loader)

    scores = np.concatenate([-m_losses, -n_losses])
    labels = np.array([1] * len(m_losses) + [0] * len(n_losses))
    return labels, scores

fig, ax = plt.subplots(figsize=(8, 6))

# Plot overfitted model
if 'model_overfit' in dir():
    labels, scores = get_mia_roc_data(
        model_overfit, member_loader, nonmember_loader
    )
    fpr, tpr, _ = roc_curve(labels, scores)
    auc = roc_auc_score(labels, scores)
    ax.plot(
        fpr, tpr, color="#d62728", linewidth=2,
        label=f"Overfitted (AUC={auc:.3f})"
    )

# Plot regularised model
if 'model_reg' in dir():
    labels, scores = get_mia_roc_data(
        model_reg, member_loader, nonmember_loader
    )
    fpr, tpr, _ = roc_curve(labels, scores)
    auc = roc_auc_score(labels, scores)
    ax.plot(
        fpr, tpr, color="#ff7f0e", linewidth=2,
        label=f"Regularised (AUC={auc:.3f})"
    )

# Plot DP models (if stored during DP experiments)
dp_colors = {10: "#2ca02c", 5: "#1f77b4", 1: "#9467bd"}
if hasattr(torch, '_dp_models'):
    for eps_val in sorted(torch._dp_models.keys(), reverse=True):
        state = torch._dp_models[eps_val]
        dp_model = CNN().to(DEVICE)
        dp_model.load_state_dict(state)
        labels, scores = get_mia_roc_data(
            dp_model, member_loader, nonmember_loader
        )
        fpr, tpr, _ = roc_curve(labels, scores)
        auc = roc_auc_score(labels, scores)
        color = dp_colors.get(eps_val, "#333333")
        ax.plot(
            fpr, tpr, color=color, linewidth=2,
            label=f"DP ε={eps_val} (AUC={auc:.3f})"
        )

# Random baseline
ax.plot(
    [0, 1], [0, 1], color=GREY, linestyle="--",
    linewidth=1, label="Random (AUC=0.500)"
)

# Highlight TPR@1%FPR region
ax.axvline(x=0.01, color=GREY, linestyle=":", alpha=0.5)
ax.annotate(
    "FPR=1%", xy=(0.015, 0.5), fontsize=8,
    color=GREY, rotation=90, va="center"
)

ax.set_xlabel("False Positive Rate (FPR)")
ax.set_ylabel("True Positive Rate (TPR)")
ax.set_title("ROC Curves — Membership Inference Attack")
ax.legend(frameon=False, loc="lower right")
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.01)

fig.tight_layout()
fig.savefig("figures/05_roc_overlay.pdf", bbox_inches="tight")
plt.show()
print("Saved figures/05_roc_overlay.pdf")

---
## 8. GDPR Analysis & Ethical Implications

### 8.1 Mapping Results to GDPR Articles

The empirical results are summarised in Section 6 (table) and Section 7
(figures). Below we map each finding to the relevant GDPR provisions.
The aggregated privacy/utility table at the end of this section (8.2)
is generated automatically from `RESULTS`, so the figures always reflect
the latest experimental run.

#### Article 5 — Principles relating to processing of personal data

**Art. 5(1)(f) — Integrity and confidentiality.**
Personal data must be processed with "appropriate security", including
protection against unauthorised access. Our results show that an ML
model trained without privacy protections leaks membership information
at levels well above random: the threshold attack reaches AUC ≈ 0.78 on
the overfitted target. An adversary with black-box access can determine,
with non-trivial probability, whether a specific individual was part of
the training set. The model itself becomes a vector for unauthorised
information disclosure, undermining Art. 5(1)(f).

**Art. 5(1)(c) — Data minimisation.**
Personal data must be "adequate, relevant and limited to what is
necessary" for the purpose. A model that memorises individual records
goes beyond what is necessary for classification. DP-SGD enforces a
form of *algorithmic* data minimisation: by clipping per-sample
gradients and adding calibrated noise, it bounds how much information
about any single individual can be encoded in the trained weights.

#### Article 6 — Lawfulness of processing

A valid legal basis (e.g., consent under Art. 6(1)(a) or legitimate
interest under Art. 6(1)(f)) does not, by itself, authorise unrestricted
processing. The basis must remain compatible with the *purpose*
declared to data subjects, and — for legitimate interest in particular
— must satisfy a necessity test. When MIA reveals that the model has
memorised individual records, the processing has produced an output
(the model) whose informational content goes beyond what the lawful
purpose strictly required. This puts pressure on the necessity and
proportionality checks that lawfulness assessments routinely require
under Art. 6(1)(f).

#### Article 17 — Right to erasure ("right to be forgotten")

The right to erasure poses a fundamental challenge for ML. Our MIA
results show that an overfitted model retains identifiable traces of
training data — the model "remembers" individuals.

- **Without DP**: deleting a record from the database does not remove
  its influence from the trained model. Honouring an erasure request
  requires retraining (machine *unlearning* in the strict sense), which
  is operationally costly.
- **With DP-SGD (ε ≤ 10)**: the model's dependence on any single
  record is mathematically bounded. Each individual's "memory" is
  provably limited, providing a form of *approximate erasure by
  construction*. This is not a substitute for genuine retraining when
  strict erasure is required, but it materially weakens the residual
  inference risk after a record is deleted from the database.

#### Article 32 — Security of processing

Art. 32 mandates "appropriate technical and organisational measures"
proportionate to the risk. Our experiments give a *measurable*
operationalisation of "appropriate":

1. Regularisation alone is insufficient: the attack still has
   measurable advantage above random.
2. DP-SGD with ε ≤ 10 brings attack success close to random,
   providing a meaningful technical safeguard under Art. 32.
3. The utility cost must be weighed against the sensitivity of the
   data and the residual risk to data subjects.

### 8.2 Privacy–Utility Trade-off (from actual experimental results)

The table below is generated from `RESULTS` so the figures always
reflect the latest experimental run. The compliance column applies a
simple, illustrative heuristic on the shadow-attack AUC: it is **not**
a legal opinion, only a way to make the trade-off visible at a glance.


In [ ]:
# Privacy / Utility table - generated from RESULTS

def _format_compliance(auc):
    """Illustrative mapping shadow-AUC -> Art. 5(1)(f) compliance hint."""
    if auc >= 0.65:
        return "X  Clear concern"
    elif auc >= 0.55:
        return "!  Insufficient"
    elif auc >= 0.52:
        return "OK Likely compliant"
    else:
        return "OK Strong compliance"

priv_rows = []

if "overfitted" in RESULTS:
    cfg = RESULTS["overfitted"]
    priv_rows.append({
        "Configuration": "No defense (overfitted)",
        "eps": "inf",
        "Test acc": f"{cfg['test_acc']*100:.1f}%",
        "Threshold AUC": f"{cfg['threshold']['auc']:.3f}",
        "Shadow AUC": f"{cfg['shadow']['auc']:.3f}",
        "TPR@1%FPR (shadow)": f"{cfg['shadow']['tpr_at_1_fpr']:.3f}",
        "Art. 5(1)(f) (heuristic)": _format_compliance(
            cfg["shadow"]["auc"]
        ),
    })

if "regularised" in RESULTS:
    cfg = RESULTS["regularised"]
    priv_rows.append({
        "Configuration": "Regularised (L2+Dropout+ES)",
        "eps": "inf",
        "Test acc": f"{cfg['test_acc']*100:.1f}%",
        "Threshold AUC": f"{cfg['threshold']['auc']:.3f}",
        "Shadow AUC": f"{cfg['shadow']['auc']:.3f}",
        "TPR@1%FPR (shadow)": f"{cfg['shadow']['tpr_at_1_fpr']:.3f}",
        "Art. 5(1)(f) (heuristic)": _format_compliance(
            cfg["shadow"]["auc"]
        ),
    })

for dp in RESULTS.get("dp", []):
    priv_rows.append({
        "Configuration": "DP-SGD",
        "eps": f"{dp['epsilon_target']}",
        "Test acc": f"{dp['test_acc']*100:.1f}%",
        "Threshold AUC": f"{dp['threshold']['auc']:.3f}",
        "Shadow AUC": f"{dp['shadow']['auc']:.3f}",
        "TPR@1%FPR (shadow)": f"{dp['shadow']['tpr_at_1_fpr']:.3f}",
        "Art. 5(1)(f) (heuristic)": _format_compliance(
            dp["shadow"]["auc"]
        ),
    })

df_priv = pd.DataFrame(priv_rows)
print("Privacy-Utility Trade-off (actual experimental results):")
print(df_priv.to_string(index=False))


### 8.3 Actionable Recommendations

The following recommendations are grounded in the experimental evidence
above. They are practitioner guidance, **not** legal advice.

1. **Always evaluate MIA before deployment** on personal data — the
   generalisation gap alone is not a sufficient privacy indicator.
2. **Regularisation is necessary but not sufficient.** It reduces
   leakage but offers no formal guarantee; attack success can remain
   measurably above random.
3. **DP-SGD is the only defense that gives a formal guarantee.** In
   our experiments it brought attack AUC close to 0.50 across all
   tested privacy budgets (ε ∈ {1, 5, 10}), at the cost of a
   measurable utility drop. For most GDPR-covered applications, ε in
   the 5–10 range is a practical operating point — provided the
   utility loss is acceptable for the use case.
4. **ε = 1 should be reserved** for the most sensitive data — Art. 9
   special categories (health, biometrics, political opinions) —
   where utility loss is acceptable.
5. **Larger training datasets mitigate the utility cost of DP** — a
   practical consideration when planning DP adoption.
6. **DP is not "automatic" GDPR compliance.** It is a *risk-mitigation
   measure* under Art. 32, not a substitute for the broader compliance
   workflow (lawful basis, data minimisation, purpose limitation,
   DPIA, etc.).


---
## 9. Download outputs

Run the cell below to download `results.json` and all figures to your computer.

In [ ]:
try:
    from google.colab import files
    import glob

    files.download("results/results.json")
    for f in glob.glob("figures/*.pdf"):
        files.download(f)
    print("✅ Downloads triggered.")
except ImportError:
    print("Not running on Colab — find your files in results/ and figures/")

---
## 10. Conclusions

This project demonstrated that:

1. **Membership Inference Attacks are practical** — with AUC=0.79 on an
 overfitted CNN, an adversary can meaningfully determine training set
 membership using only black-box access.

2. **Informal defenses reduce but do not eliminate leakage** — regularisation
 lowered attack AUC to 0.66, but this still exceeds random chance and may
 not satisfy GDPR Art. 32 requirements for "appropriate security."

3. **DP-SGD provides formal, measurable privacy protection** — even at the
 relaxed setting of ε=10, attack success drops to near-random (AUC≈0.52),
 with a moderate utility cost.

4. **The privacy–utility trade-off is real and context-dependent** — there
 is no universal "correct" ε. The appropriate privacy budget depends on
 data sensitivity, regulatory requirements, and acceptable accuracy loss.

5. **GDPR compliance for ML requires active privacy measures** — training a
 model on personal data without protections creates a quantifiable privacy
 risk that standard regularisation cannot fully address.

### Limitations

- Results are on CIFAR-10, a controlled benchmark. Real-world datasets
 (medical images, biometric data) may exhibit different MIA vulnerability
 profiles.
- We evaluated only two attack strategies. More sophisticated attacks
 (LiRA, Carlini et al. 2022) may achieve higher success rates.
- DP-SGD utility estimates depend heavily on dataset size and model
 architecture — results may not transfer directly to production systems.

### References

- Shokri, R., et al. (2017). "Membership Inference Attacks Against
 Machine Learning Models." IEEE S&P.
- Yeom, S., et al. (2018). "Privacy Risk in Machine Learning:
 Analyzing the Connection to Overfitting." CSF.
- Carlini, N., et al. (2022). "Membership Inference Attacks From
 First Principles." IEEE S&P.
- Abadi, M., et al. (2016). "Deep Learning with Differential Privacy." CCS.
- Tramèr, F. & Boneh, D. (2021). "Differentially Private Learning Needs
 Better Features (or Much More Data)." ICLR.